# Writing client code: notebooks and apps

This notebook is for whoever writes code against `ess.apps`: a notebook, a local app, a service-backed app, or a program.
It follows one thread, from a for loop over runs to a sum of runs that grows as they arrive.
Every step shows what the framework guarantees, and what the client must do in return.

The workflows are the toy SANS reduction of `toy_sans.py`.
A run's file holds counts per pixel and a monitor count.
`IOFQ` reduces one sample run and one can run to an I(Q) curve, `IOFQ_MULTI` sums several of each, and `FORWARD` reads a curve and returns its first point.
`FIND_CENTRE` finds the beam centre from a sample run, and `TRANSMISSION` computes a sample's transmission from its transmission run.
[workflow-authors.ipynb](workflow-authors.ipynb) shows how these specs and their bindings are written.
Here they are given, as an installed package would give them.

A spec has *params*, everything the workflow takes, and *outputs*.
Each field of the params model is a *parameter*.
A request gives the value of each field, at any depth, in one of three forms:
*written inline*, the value itself, such as `Reduction(bins=4)`;
a *file*, such as `file(run=101)`;
or a *reference to an output*, which names an output of a record or of an accumulator.
The record of a request stores what the request gives in *JSON form*, what it dumps to as JSON: the *stored form*.
The type of a field says which forms it takes.
A *group*, a pydantic model such as `Reduction` or `BeamCentre`, is written inline as a JSON object of its fields.
A *file field*, typed `Annotated[Path, File(...)]` such as `RunFile`, takes a file.
A field typed `Referenced[T]` takes a reference to an output whose data fit `T`.
A group with a field that no request can write, such as the scipp variable of `IofQ`, is given by reference only, at a field typed `Referenced[IofQ]`.
A field typed `BeamCentre | Referenced[BeamCentre]` takes either.
[params.md](../../../packages/essspec/docs/params.md) states the rules.

| Spec | Params | Outputs |
|---|---|---|
| `IOFQ` | `reduction: Reduction` (`bins=2`), `beam_centre: BeamCentre \| Referenced[BeamCentre]` (`x=0, y=0`), `transmission: TransmissionFraction \| Referenced[TransmissionFraction]`, `sample: SampleRunFile`, `can: BackgroundRunFile` | `iofq: IofQ` |
| `IOFQ_MULTI` | `reduction`, `beam_centre`, and `transmission` as in `IOFQ`, `sample: list[SampleRunFile]`, `can: list[BackgroundRunFile]` | `iofq: IofQ` |
| `FORWARD` | `curve: Referenced[IofQ]` | `i0: sc.Variable` |
| `FIND_CENTRE` | `sample: SampleRunFile` | `centre: BeamCentre` |
| `TRANSMISSION` | `run: TransmissionRunFile` | `transmission: TransmissionFraction` |

By convention a parameter is a group, a union of groups, a file field, or a list of one.
A form shows a group as one tab, and a binding sets it as one key.
A file parameter needs `keys=` in its binding.
A parameter that takes a file is a file field: `ess.spec` provides `RunFile`, which types a run's raw file, a type per role of a run, such as `SampleRunFile`, for which a picker offers sample runs and runs without a role, and `MaskFile`, for a mask; the workflow gets the file's local path.
`IofQ` is a group whose field `intensity` is the curve, a scipp variable, and `i0` is a scipp variable.
`transmission` has no default: a request gives a transmission, typed in, `{'value': 0.9}`, or computed by `TRANSMISSION`, and its record says which.
`IOFQ_MULTI` has the settings of `IOFQ`, a table of sample runs, and a table of can runs (section 6); the summed sample runs have one transmission.

In [ ]:
import tempfile
from pathlib import Path

from ess.apps import (
    Client,
    NotReadable,
    Request,
    SubmitError,
    Template,
    Workspace,
    file,
)
from ess.apps.testing import FakeFiles, FakeStore
from toy_sans import (
    BINDINGS,
    CAN_RUNS,
    FIND_CENTRE,
    FORWARD,
    IOFQ,
    IOFQ_MULTI,
    SAMPLE_RUNS,
    TRANSMISSION,
    TRANSMISSION_RUNS,
    BeamCentre,
    Reduction,
    TransmissionFraction,
)

## Setup

The framework reads raw data through a *file source*.
Here a fake one writes a few runs of proposal `p1` as files in a temporary folder, and a fake store stands in for disk.
A *workspace* holds the history and the output values of one proposal, and runs its requests.
A *client* is a connection to a workspace and keeps nothing: every client of a workspace reads the same values.

In [ ]:
folder = tempfile.TemporaryDirectory()
files = FakeFiles(proposal='p1', directory=Path(folder.name))
for run, data in SAMPLE_RUNS.items():
    files.measure(run, data, role='sample', temperature=f'{run + 149}K')
for run, data in CAN_RUNS.items():
    files.measure(run, data, role='background')
for run, data in TRANSMISSION_RUNS.items():
    files.measure(run, data, role='transmission')

workspace = Workspace(proposal='p1', files=files, bind=BINDINGS, store=FakeStore())
client = Client(workspace, submitter='alice')

`local(proposal='p1', files=files, bind=BINDINGS, store=...)` does the same in one call.
This notebook keeps `workspace` to open a second client later, as another process would with `connect(url, proposal='p1')` on the service.
The service is not implemented; everything below runs in this process.

## 1. From a for loop to records

A notebook without the framework calls the reduction directly:

```python
curves = {}
for run in (101, 102, 103):
    curves[run] = reduce_iofq(sample=run, can=110, bins=2)
```

With the framework, each call becomes a request: a spec and its params, by name.
`file(run=101)` names a file; every parameter but the runs and the transmission has a default, so the request leaves them out.
The sample and the can are each given by a file, and the transmission here is typed in, `{'value': 0.8}`.
`submit` returns a *record* at once, and the requests run in parallel.

In [ ]:
can = file(run=110)
transmission = {'value': 0.8}
records = {}
for run in (101, 102, 103):
    params = {'sample': file(run=run), 'can': can, 'transmission': transmission}
    records[run] = client.submit(IOFQ, params=params)
client.wait(records)
client.output(records[101], 'iofq')

A record holds the request as the framework accepted it: every parameter filled in, defaults included, and each file resolved from the name the user gave (`run:101`) to its identity.
It stores the group `transmission` as an object of its fields.
It holds no output values.
A record never changes; only its status moves from `pending` to `completed`, `failed`, or `cancelled`.

In [ ]:
record = records[101]
record.request.params, client.status(record)

Every call takes one record, a list, or a dict, and returns the same shape:

In [ ]:
client.status(records)

**A request that cannot run is refused at submission**, with an error that names the field.
Each of its `errors` names the request by its key, `entry`, the field by its position, `loc`, and says why, `msg`.
If one request of a call is refused, none is submitted, so a batch with one typo starts nothing.

In [ ]:
params = {'sample': file(run=102), 'can': can, 'transmission': transmission}
batch = {
    'good': Request(IOFQ, params=params),
    'typo': Request(IOFQ, params={'reduction': {'bins': 0}, **params}),
}
try:
    client.submit(batch)
except SubmitError as error:
    print(error)
    for e in error.errors:
        print(e.entry, e.loc, e.msg)

A group may be given as a dict of its fields, as above, or as an instance of its model, which a type checker checks (section 2); both make the same request.
A request may also give its params as an instance of the params model, if it holds no reference to an output; each `Path` in it names a file by its path, as `file(path=...)` does (see `Client.submit`).
`Request(spec, params=...)` makes a request without submitting it, so that several go into one call.

A request gives a file as `file(...)`: by run, by PID, or by path, `file(path=...)`.
The workspace resolves it at submission, so that the file source checks access and every file a request reads has an identity in the record; a path given as a string is refused:

In [ ]:
try:
    client.submit(
        IOFQ,
        params={
            'sample': '/data/run-102.json',
            'can': can,
            'transmission': transmission,
        },
    )
except SubmitError as error:
    print(error)

## 2. Chains: a reference to a pending record

A value may also be an output of a record, given as `record.ref(output)`.
The record may still be pending, so a chain is submitted without waiting.
The framework runs `FORWARD` once the I(Q) has completed.

In [ ]:
curve = client.submit(
    IOFQ,
    params={
        'reduction': Reduction(bins=4),
        'sample': file(run=103),
        'can': can,
        'transmission': transmission,
    },
)
i0 = client.submit(FORWARD, params={'curve': curve.ref('iofq')})
client.output(i0, 'i0')

`client.output` waits for the record.
The framework checks at submission that the output *fits* the parameter, by their types: an `IofQ` output fits a parameter typed `Referenced[IofQ]`.
A reference to an output fills a field typed `Referenced[T]`, with `T` a group, since a group's class says what the value is: a field typed `sc.Variable` would take every output of that type, so a picker of records would list unrelated outputs.
The output `i0` of `FORWARD`, a scipp variable, does not fit `IofQ`:

In [ ]:
try:
    client.submit(FORWARD, params={'curve': i0.ref('i0')})
except SubmitError as error:
    print(error)

A field typed `BeamCentre | Referenced[BeamCentre]`, such as `beam_centre`, takes a value written inline or a reference, whichever the request gives.
A field without a member `Referenced[...]`, such as `reduction`, takes no reference.
One user types in the beam centre; another computes it with `FIND_CENTRE`, whose output `centre` is a `BeamCentre`:

In [ ]:
finder = client.submit(FIND_CENTRE, params={'sample': file(run=103)})
runs = {'sample': file(run=103), 'can': can, 'transmission': transmission}
centred = {
    'inline': client.submit(
        IOFQ, params={'beam_centre': BeamCentre(x=1.25, y=0.0), **runs}
    ),
    'reference': client.submit(
        IOFQ, params={'beam_centre': finder.ref('centre'), **runs}
    ),
}
print(client.output(finder, 'centre'))
{form: client.output(r, 'iofq').intensity.values for form, r in centred.items()}

The workflow gets a `BeamCentre` either way, and the finder found (1.25, 0), so both give the same I(Q).
Each record stores the value in the form the request gave it, inline or as a reference, so the record of the second says where its beam centre came from:

In [ ]:
[centred[form].request.params['beam_centre'] for form in centred]

## 3. Who keeps a value

An output value, such as the I(Q) array, lives while something keeps it, and nothing else keeps it:

| Keeper | Keeps the value until |
|---|---|
| the workspace, for a record submitted without persisting its outputs | a client releases the record, or the workspace closes |
| a pending request that reads it | that request's workflow has returned |
| a stage, an accumulator's template, or a push that references it | the binding has taken it, or the stage, accumulator, or push is released or stopped |
| the store, once the value is written | the proposal's history is dropped |

A record object, a reference, or a label keeps nothing.
Releasing drops the workspace's hold; the record stays, but its value is gone for good:

In [ ]:
client.release(curve)
try:
    client.output(curve, 'iofq')
except NotReadable as error:
    print(error)

`FORWARD` had read it already, so `i0` is unaffected.
Had `FORWARD` still been pending, it would have kept the curve until it ran.

Two consequences that a client author must know:

- **Work that nothing keeps is cancelled.** Releasing a pending record that no pending request reads cancels it, and closing the workspace releases everything it keeps. Closing the client of `local()` closes its workspace, so a notebook that submits a batch and closes that client before the batch has run cancels the batch, unless it was persisted.
- **Every client of the workspace reads the same values**, and a release by one client is a release for all. Closing a client made with `Client(workspace, ...)` releases nothing.

A workspace keeps a record's values until a client releases the record, so a long-lived app releases what it no longer shows.
On the service, how the memory of a workspace is bounded, and when an idle workspace is ended, are open.

## 4. Persist, and find it the next day

`persist=` at submission makes the store the only keeper.
This is how an unattended batch is submitted: the workspace keeps nothing, and the records complete once their outputs are written.
A *label* names the batch, and a dict's keys become the *members*:

In [ ]:
night = client.submit(
    {
        f'{run + 149}K': Request(
            IOFQ,
            params={
                'sample': file(run=run),
                'can': can,
                'transmission': transmission,
            },
        )
        for run in (101, 102, 103)
    },
    label='scan',
    persist=True,
)
client.wait(night)

The next morning, another client of the same proposal finds the batch by its label and reads from the store.
Here a second client of the same workspace stands in for it:

In [ ]:
morning = Client(workspace, submitter='bob')
latest = morning.latest('scan', member='250K')
morning.output(latest, 'iofq')

On the service, records are the proposal's history: they last until the proposal has been idle for days to weeks, and are then dropped as a whole.
With `local()`, history is held in memory and ends with the process.
What must last longer is published to SciCat (not implemented).

## 5. Templates: tune with a stage

A *template* is a spec with some parameters fixed, and *blanks*, the names of the parameters left open.
It takes the fixed values as a request does, as `params=`.
A *stage* is a template the workspace keeps, so that what depends only on the fixed values is computed once.
Tuning `bins` then does not load the runs again:

In [ ]:
tune = client.stage(
    Template(
        IOFQ,
        params={
            'sample': file(run=101),
            'can': can,
            'transmission': transmission,
        },
        blanks=('reduction',),
    )
)
for bins in (1, 2, 4):
    tuned = client.compute(tune, params={'reduction': Reduction(bins=bins)})
    print(bins, client.output(tuned, 'iofq').intensity.values)

A call through a stage makes the record of the plain request, the one with every value given.
Nothing in the record says it went through a stage, and its outputs are those of the plain request (the *caching* symmetry):

In [ ]:
tuned.request

A blank is a parameter, never a field of a group, and a parameter is one key of the workflow's pipeline.
The transmission is a parameter of its own, so a stage that tunes it has `transmission` as its blank, and loads no run again.

A stage needs a binding that makes stages, such as one built from a sciline pipeline, which computes the fixed part once.
A stage over any other binding is refused when it is made:

In [ ]:
try:
    client.stage(Template(FORWARD, blanks=('curve',)))
except SubmitError as error:
    print(error)

## 6. Sum runs as they arrive: an accumulator

A *table* is a list parameter whose *rows* an accumulator takes a push at a time.
`IOFQ_MULTI` has two tables: `sample`, whose rows are sample runs, and `can`, whose rows are can runs ([accumulators.md](../accumulators.md#accumulator)).

An *accumulator* is a template whose blanks are tables, kept by the workspace.

Each table is either a blank or fixed.
A fixed table is given whole in `params`, as a request gives it, and stays as given for the accumulator's life.
With one can for all samples, the template fixes `can` and leaves `sample` blank:

```python
Template(
    IOFQ_MULTI,
    params={'can': [can], 'transmission': transmission},
    blanks=('sample',),
)
```

A push into a fixed table is refused, and `client.accumulator` refuses a blank that is not a table.

Each *push* gives rows of one or more tables, by the name of each, as a list.
State n, the accumulator after n pushes, has the outputs of the plain request whose tables hold the rows of the first n pushes.

In [ ]:
measured = client.submit(TRANSMISSION, params={'run': file(run=120)})
total = client.accumulator(
    Template(
        IOFQ_MULTI,
        params={'transmission': measured.ref('transmission')},
        blanks=('sample', 'can'),
    )
)
total.push(can=[can])  # state 1
total.push(sample=[file(run=101)])  # state 2
state_2 = client.output(total, 'iofq').intensity
total.push(sample=[file(run=102)])  # state 3
state_3 = client.output(total, 'iofq').intensity
state_2.values, state_3.values

`push` returns once the workspace has logged it. The workspace adds it later.
A read is of the state after the pushes logged so far, and waits for it.
State 1 has no sample row, so a read of it would be refused: the table `sample` needs a row.

A value that differs per run is in the run's file.
The summed sample runs have one transmission, a parameter of its own: here the template gives it by reference to the transmission computed from transmission run 120.

State 3 equals the plain request over the same rows (the *arrival* symmetry):

In [ ]:
plain = client.compute(
    IOFQ_MULTI,
    params={
        'sample': [file(run=101), file(run=102)],
        'can': [can],
        'transmission': TransmissionFraction(value=0.9),
    },
)
client.output(plain, 'iofq').intensity.values

With one row in each table, a request of `IOFQ_MULTI` gives the outputs of the request of `IOFQ` with the same values (the *one-row* symmetry).

Behind the accumulator, the binding of `IOFQ_MULTI` makes a *driver*, which the workspace keeps for the accumulator.
The driver holds the sums of numerators and denominators, and a push loads its runs and adds them.
Only a spec whose binding makes drivers has accumulators; the workspace refuses any other when the accumulator is made.

A push may give rows of several tables, which enter one state, as the push of runs 103 and 111 below does.
The tables stand side by side, as in a request: rows of different tables have nothing to do with each other, and the first sample row is not paired with the first can row.
A push may give several rows of its table, which enter one state; a push of no rows is refused.

**Readers never see a value change.**
`client.output` of an accumulator returns a copy, so `state_3` above still holds state 3 once push 4 is added.
The workspace adds push 4 only once the reads of state 3 have copied:

In [ ]:
total.push(
    sample=[file(run=103)],
    can=[file(run=111)],
)  # state 4
client.output(total, 'iofq').intensity.values, state_3.values

The order of the calls is the order of the states, as if each call ran after the one before it, although pushes and reads run in parallel.

The rows of a table are distinct.
A row pushed again is refused, since pushing a run twice is usually a cell run twice.
Rows are compared by their stored form: a run by its file's identity, so run 103 named by its path is the same row.
A reference to an output is compared by its record and output, so two references to equal data are two rows.

In [ ]:
try:
    total.push(sample=[file(path=Path(folder.name) / 'run-103.json')])
except SubmitError as error:
    print(error)

At the end, `freeze` turns the last state into an ordinary record, without a copy, and the accumulator takes no more pushes.
A request never references an accumulator.
It references the record of the freeze, as it references any record:

In [ ]:
summed = client.freeze(total)
forward_4 = client.submit(FORWARD, params={'curve': summed.ref('iofq')})
client.output(summed, 'iofq').intensity.values, client.output(forward_4, 'i0').value

## 7. Where a result came from

Every record can say which requests and files produced it.
The record of a frozen state holds the plain request over its rows, each run as its file's identity, and does not name the accumulator.
The transmission is a reference to the record of `TRANSMISSION`:

In [ ]:
provenance = client.provenance(summed)
print(summed.request.params['transmission'])
[str(d) for d in provenance.files()]

## 8. Flavours of client code

Everything above is the same API for every client.
What differs between flavours is which calls they rely on, and what they must take care of.

| Flavour | Makes its client with | Typically | Must take care of |
|---|---|---|---|
| notebook | `local(...)` | `submit`, `output`, stages, accumulators | releasing large values; not modifying values that `output` returned (see below) |
| local app (not a notebook) | `local(...)` | the same, behind a form | releasing what it no longer shows; choosing the rows of each table |
| app against the service | `connect(url, proposal=...)` (not implemented) | `submit(..., persist=True)`, labels, `latest` | releasing what it no longer shows (how the service bounds a workspace's memory is open); reading results from the store |
| automatic reduction | either | `Rule`, `TriggerLoop` | choosing what to persist |
| an app for many samples | `local(...)` | `ess.apps.batch` | see its [guide](../../../packages/essapps/docs/batch.md) |
| program, CLI, agent | `connect(...)` (not implemented) | `SpecId`, `SerializedWorkflowSpec` | importing no workflow package |

Points that hold for apps but not for notebooks:

- **Forms come from the spec.** `IOFQ.serialize()` gives the schema of the params: the JSON Schema of each group that a request writes, for a form, and the type of data that fits each file or reference, such as `{'kind': 'raw'}`, the metadata of the files to offer for a run's file, or `BeamCentre` at `Referenced[BeamCentre]`, for a picker of files or records. An app that imports only a package's specs, kept in a module apart from the workflow code, builds forms without the workflow's dependencies ([ADR 0007](../adr/0007-packages-split-by-dependencies.md)).
- **Errors name the field at fault.** Each of the `errors` of a `SubmitError` is a `FieldError`: `entry`, the key or index of the request in the call (`'typo'` above), `loc`, the position of the field (`('reduction', 'bins')`), and `msg`. A form shows each next to its field. A read raises `NotReadable` for a record or an accumulator that exists but cannot be read, such as one released, `KeyError` for one it does not know, and `RecordFailed` for a record that failed or was cancelled.
- **An app that does not import the workflow package** names the spec by `SpecId(name='toy-sans-iofq', version=1)` and gets the same checks.
- **The framework knows no technique.** Which sample run goes with which can run, and which runs form the rows of a table, is the app's decision.

**A batch** is a template filled once per file and submitted as a dict of requests, here keyed by a metadata field. Each file fills a blank, a file parameter, here the sample run whose beam centre `FIND_CENTRE` finds:

In [ ]:
from ess.apps import Selector

samples = client.files.list(Selector(role='sample'))
template = Template(FIND_CENTRE, blanks=('sample',))
requests = {
    str(client.files.metadata(run)['temperature']): template.fill(params={'sample': run})
    for run in samples
}
batch = client.submit(requests, label='batch', persist=True)
client.wait(batch)
sorted(r.member for r in client.records(label='batch'))

A `Rule` is a name, a function, a selector, a label, and the parameter `at` that holds the file. The `TriggerLoop` calls the function for each new file that the selector matches; the function pairs runs as the technique needs, submits what the file needs, and returns the record that settles the file. See [automatic-reduction.md](../automatic-reduction.md).

**Values in the user's process are not copied.**
In a notebook, `client.output` of a record returns the value the framework holds, not a copy.
Modifying it in place changes what later requests read.
This is open question 7 of the [README](../README.md); until it is settled, treat outputs as read-only.

## Summary: what client code relies on, and what it owes

| Relies on | Owes |
|---|---|
| a record never changes, only its status | persisting at submission what must outlive the workspace, and releasing what no client still needs |
| a refused call submits nothing, and names the field | treating output values as read-only |
| a value lives while something keeps it, and only then | giving each file as `file(...)`, which the file source resolves |
| a request runs once the records it references have completed | choosing the rows of each table: the framework knows no technique |
| a value is written inline, a file, or a reference to an output, as the spec types its field; a reference to an output is checked by type at submission, and the workflow gets the data it names | |
| a reader never sees a value change | |
| a call through a stage, or a state of an accumulator, gives the outputs of its plain request | |
| where a request runs changes neither its record nor its outputs | |

[framework-promises.ipynb](framework-promises.ipynb) shows what keeping these promises takes, in this process and on the service.

In [ ]:
client.close()
morning.close()
workspace.close()
folder.cleanup()